# Faza 2 — Puna naracija „Šta je AI" (original + prerušene verzije)

Generiše **celu** naraciju videa 001 tvojim kloniranim glasom, u više verzija:
- `shift 0.0` = tvoj glas (original klon)
- `shift -0.25`, `-0.5` = referenca pomerena pre kloniranja → **čist** prerušen glas (bez artefakata pomeranja gotovog snimka)

Slušaš dugu naraciju i biraš koja ti se najviše sviđa. Sve se snima na Drive (`ai-glas/narration_*.wav`).

> Prvo: Runtime → GPU. Recept za instalaciju je isti kao u `fish_speech_clone_test.ipynb` (portaudio + torchvision 0.23.0 + transformers 4.57.3).

## 1. Setup (GPU, Drive, instalacija, model)

In [ ]:
!nvidia-smi -L
from google.colab import drive; drive.mount('/content/drive')

In [ ]:
%cd /content
!git clone https://github.com/fishaudio/fish-speech.git 2>/dev/null || echo '(repo vec postoji)'
%cd /content/fish-speech
!apt-get -qq install -y portaudio19-dev
!pip install -e . -q
!pip install -q torchvision==0.23.0 "transformers==4.57.3"
import torch; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())

In [ ]:
import os, subprocess
MODEL_DIR='checkpoints/openaudio-s1-mini'
CODEC=f'{MODEL_DIR}/codec.pth'
for cli in (['huggingface-cli','download'], ['hf','download']):
    r=subprocess.run(cli+['fishaudio/openaudio-s1-mini','--local-dir',MODEL_DIR], capture_output=True, text=True)
    print(cli[0],'RC',r.returncode)
    if r.returncode==0: break
    print(r.stderr[-800:])
assert os.path.exists(CODEC), f'NEMA {CODEC} — download nije uspeo (gore je razlog). Ako je gated: prihvati uslove na HF + login().'
print('model spreman:', os.listdir(MODEL_DIR))

## 2. Parametri: referenca, transkript, naracija, pomaci

`PROMPT_TEXT` mora da odgovara prvih ~22s tvog uzorka (isto kao u clone testu). `SHIFTS` su pomaci u polutonima (0 = original). Dodaj/izbaci po želji.

In [ ]:
REF_FULL = '/content/drive/MyDrive/ai-glas/owner-sample.wav'
SHIFTS = [0.0]   # prvo SAMO trenutni glas; kasnije dodaj npr. -0.5 ako odlucis da prerusis

# Tacne reci iz prvih ~22s tvog uzorka (ISPRAVI ako treba):
PROMPT_TEXT = "Veštačka inteligencija danas više nije nešto što gledamo samo u filmovima. Ona piše tekst, pravi slike, pomaže u programiranju i odgovara na pitanja iz skoro svake oblasti."

# Cela naracija videa 001 (iz content/001-sta-je-ai/script.json):
NARRATION = """Veštačka inteligencija u 2026. više nije naučna fantastika. Danas ti piše kod, pravi slike i odgovara na pitanja bolje nego ikad. Za nekih sedam minuta objasniću ti šta ona zaista jeste, šta sve može i kako da je koristiš potpuno besplatno.
Ako se prvi put susrećeš sa ovim svetom, opusti se. Sve ćemo proći redom, jednostavnim rečima, bez nepotrebnih komplikacija.
Krenimo od osnovnog pitanja: šta je zapravo veštačka inteligencija? Kada danas kažemo AI, najčešće mislimo na takozvane velike jezičke modele. To su programi koji su trenirani na ogromnoj količini teksta i koda. Iz svega što su pročitali, oni nauče da predvide koja reč najverovatnije dolazi sledeća.
Zvuči jednostavno, ali baš iz tog predviđanja sledeće reči izranja nešto moćno. Model može da napiše ceo tekst, da odgovori na pitanje ili da reši zadatak. Važno je da zapamtiš jedno: model ne razmišlja kao čovek. On veoma dobro pogađa na osnovu obrazaca koje je naučio.
Hajde da vidimo to u praksi. Otvorim alat, ukucam pitanje običnim jezikom, i za par sekundi dobijem jasan odgovor. Isto tako mogu da tražim da mi napiše kod, da mi skrati dugačak tekst ili da mi objasni pojam koji ne razumem.
A ko pravi te modele? U 2026. tri imena se najčešće pominju. Anthropic, koji stoji iza Claude-a. OpenAI, poznat po GPT i Codex modelima. I Google, sa porodicom modela koja se zove Gemini. Svaki od njih ima svoje jače i slabije strane, pa se isplati probati više njih.
Dve stvari su se drastično promenile u poslednje dve godine. Prvo, modeli sada mogu da obrade mnogo više teksta odjednom. Drugo, cena korišćenja je znatno pala. To zajedno znači da je danas moćan alat dostupan skoro svakome, a ne samo velikim firmama.
Ali budimo pošteni i oko mana. Model ponekad zvuči potpuno sigurno, a zapravo greši. To zovemo halucinacija. Zato uvek proveri važne informacije i ne veruj svemu na reč. Alat je tu da ti pomogne, a ne da razmišlja umesto tebe.
Kako da počneš već danas, i to besplatno? Dovoljno je da otvoriš jedan od besplatnih alata u pregledaču i da mu postaviš prvo pitanje. Najbolji savet koji mogu da ti dam: budi jasan i konkretan. Što tačnije opišeš šta želiš, to je bolji rezultat.
Ako ti je ovo bilo korisno, prijavi se na kanal. U sledećem videu pokazujem ti kako da napišeš prompt koji stvarno daje dobre rezultate.
Hvala ti što si gledao. Vidimo se u sledećem videu."""
print('Parametri spremni. Pomaci:', SHIFTS)

## 2b. BRZA provera (jedna recenica)
Pre duge naracije — da za ~1 min znas da li okruzenje radi. Ako je SANITY RC: 0 i cujes glas, sve je u redu; idi na korak 3. Ako RC: 1, posalji mi ispis.

In [ ]:
import librosa, soundfile as sf, subprocess, IPython.display as ipd
y,sr=librosa.load(REF_FULL,sr=None,mono=True); sf.write('/content/ref_chk.wav', y[:int(22*sr)], sr)
subprocess.run(['python','fish_speech/models/dac/inference.py','-i','/content/ref_chk.wav','--checkpoint-path',CODEC], check=True)
r=subprocess.run(['python','fish_speech/models/text2semantic/inference.py','--text','Ovo je kratak test glasa.','--prompt-text',PROMPT_TEXT,'--prompt-tokens','fake.npy','--checkpoint-path',MODEL_DIR,'--num-samples','1'], capture_output=True, text=True)
print('SANITY RC:', r.returncode)
if r.returncode!=0:
    print(r.stderr[-1500:])
else:
    subprocess.run(['python','fish_speech/models/dac/inference.py','-i','codes_0.npy','--checkpoint-path',CODEC], check=True)
    print('OK — okruzenje radi!'); ipd.display(ipd.Audio('fake.wav'))

## 3. Generiši sve verzije (svaka ~3–4 min zvuka; potraje)

Za svaki pomak: pomeri referencu → enkoduj → generiši tokene iz cele naracije → dekoduj → snimi na Drive i pusti.

In [ ]:
import librosa, soundfile as sf, subprocess, shutil, os, IPython.display as ipd

DRIVE_OUT = '/content/drive/MyDrive/ai-glas'
y, sr = librosa.load(REF_FULL, sr=None, mono=True)
y = y[:int(22*sr)]   # ~22s referenca

results = []
for s in SHIFTS:
    print(f'\n===== POMAK {s} polutona =====')
    yy = y if s == 0 else librosa.effects.pitch_shift(y, sr=sr, n_steps=s)
    refp = f'/content/ref_{s}.wav'; sf.write(refp, yy, sr)
    subprocess.run(['python','fish_speech/models/dac/inference.py','-i',refp,'--checkpoint-path',CODEC], check=True)
    subprocess.run(['python','fish_speech/models/text2semantic/inference.py','--text',NARRATION,'--prompt-text',PROMPT_TEXT,'--prompt-tokens','fake.npy','--checkpoint-path',MODEL_DIR,'--num-samples','1'], check=True)
    subprocess.run(['python','fish_speech/models/dac/inference.py','-i','codes_0.npy','--checkpoint-path',CODEC], check=True)
    outp = f'{DRIVE_OUT}/narration_sta-je-ai_shift_{s}.wav'
    shutil.copy('fake.wav', outp); results.append((s, outp))
    print('Snimljeno:', outp)

print('\n\n========== PRESLUŠAJ ==========')
for s, outp in results:
    print(f'--- pomak {s} ---'); ipd.display(ipd.Audio(outp))